In [2]:
import xml.etree.ElementTree as ET
import csv

# -------- FILE PATHS --------
HMDB_XML = "hmdb_metabolites.xml"
INPUT_CSV = "SIGNOR_smallmolecular_to_check.csv"   # your file with IDA column
OUTPUT_FILE = "chebi_endogenous_results.csv"


# -------- STEP 1: LOAD & CLEAN INPUT --------
def load_chebi_ids(file_path):
    chebi_ids = set()

    with open(file_path, "r", newline="") as f:
        reader = csv.DictReader(f)

        if "IDA" not in reader.fieldnames:
            raise ValueError("Column 'IDA' not found in CSV")

        for row in reader:
            raw = row["IDA"].strip()

            if not raw:
                continue

            raw = raw.upper()  # normalize case

            # Keep only CHEBI IDs
            if raw.startswith("CHEBI:"):
                chebi_ids.add(raw)

    return chebi_ids


# -------- STEP 2: PARSE HMDB XML --------
def extract_endogenous_chebi_ids(xml_file):
    endogenous_chebi = set()

    context = ET.iterparse(xml_file, events=("end",))

    count = 0

    for event, elem in context:
        if elem.tag.endswith("metabolite"):

            count += 1
            if count % 5000 == 0:
                print(f"Processed {count} metabolites...")

            is_endogenous = False
            chebi_ids = []

            for child in elem:
                tag = child.tag.lower()

                # Check origin
                if tag.endswith("origin") and child.text:
                    if "endogenous" in child.text.lower():
                        is_endogenous = True

                # Extract ChEBI IDs
                if tag.endswith("external_identifiers"):
                    for ext in child:
                        resource = None
                        identifier = None

                        for sub in ext:
                            if sub.tag.endswith("resource"):
                                resource = sub.text
                            elif sub.tag.endswith("identifier"):
                                identifier = sub.text

                        if resource and "chebi" in resource.lower():
                            if identifier:
                                chebi_ids.append(f"CHEBI:{identifier}")

            if is_endogenous:
                endogenous_chebi.update(chebi_ids)

            elem.clear()  # free memory

    return endogenous_chebi


# -------- STEP 3: MATCH --------
def compare_ids(input_ids, endogenous_ids):
    results = []

    for cid in input_ids:
        status = "Endogenous" if cid in endogenous_ids else "Not Endogenous"
        results.append((cid, status))

    return results


# -------- STEP 4: SAVE OUTPUT --------
def save_results(results, output_file):
    # Remove duplicates again just in case
    results = list(set(results))

    # Sort nicely
    results.sort(key=lambda x: x[0])

    with open(output_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["ChEBI_ID", "Status"])
        writer.writerows(results)


# -------- MAIN --------
if __name__ == "__main__":
    print("Loading and cleaning input IDs...")
    input_ids = load_chebi_ids(INPUT_CSV)
    print(f"Unique CHEBI IDs: {len(input_ids)}")

    print("\nParsing HMDB XML (this will take a while)...")
    endogenous_ids = extract_endogenous_chebi_ids(HMDB_XML)
    print(f"\nTotal endogenous CHEBI IDs in HMDB: {len(endogenous_ids)}")

    print("\nMatching IDs...")
    results = compare_ids(input_ids, endogenous_ids)

    print("Saving results...")
    save_results(results, OUTPUT_FILE)

    print("\nDone! Output saved to:", OUTPUT_FILE)

Loading and cleaning input IDs...
Unique CHEBI IDs: 141

Parsing HMDB XML (this will take a while)...
Processed 5000 metabolites...
Processed 10000 metabolites...
Processed 15000 metabolites...
Processed 20000 metabolites...
Processed 25000 metabolites...
Processed 30000 metabolites...
Processed 35000 metabolites...
Processed 40000 metabolites...
Processed 45000 metabolites...
Processed 50000 metabolites...
Processed 55000 metabolites...
Processed 60000 metabolites...
Processed 65000 metabolites...
Processed 70000 metabolites...
Processed 75000 metabolites...
Processed 80000 metabolites...
Processed 85000 metabolites...
Processed 90000 metabolites...
Processed 95000 metabolites...
Processed 100000 metabolites...
Processed 105000 metabolites...
Processed 110000 metabolites...
Processed 115000 metabolites...
Processed 120000 metabolites...
Processed 125000 metabolites...
Processed 130000 metabolites...
Processed 135000 metabolites...
Processed 140000 metabolites...
Processed 145000 metab